In [1]:
# ============================================================
# DIAGNOSTIC — Anomalie effectif juin 2025 (GRADE A3)
# ============================================================
# Remonte la chaîne Bronze → Silver → Gold pour localiser
# à quelle étape les ~45 000 agents A3 disparaissent en juin 2025.
#
# Dépendances :
#   pip install polars boto3 python-dotenv
# ============================================================

import io
import os
import re
from dotenv import load_dotenv
import boto3
from botocore.client import Config
import polars as pl

load_dotenv(".env")

# ============================================================
# CONFIGURATION
# ============================================================

MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT",   "http://192.168.1.230:30137")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "datalab-team")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minio-datalabteam123")

BUCKET_BRONZE = os.getenv("BUCKET_BRONZE", "bronze")
BUCKET_SILVER = os.getenv("BUCKET_SILVER", "silver")
BUCKET_GOLD   = os.getenv("BUCKET_GOLD",   "gold")

PREFIX_BRONZE = "panel_admin"
PREFIX_SILVER = "panel_admin"
PREFIX_PANEL  = "panel_admin"

MOIS_CIBLE  = "062025"   # format MMYYYY du Silver
GRADE_CIBLE = "A3"

# ============================================================
# CLIENT S3
# ============================================================

s3 = boto3.client(
    "s3",
    endpoint_url          = MINIO_ENDPOINT,
    aws_access_key_id     = MINIO_ACCESS_KEY,
    aws_secret_access_key = MINIO_SECRET_KEY,
    config                = Config(
        signature_version = "s3v4",
        retries           = {"max_attempts": 5, "mode": "adaptive"},
        connect_timeout   = 30,
        read_timeout      = 120,
    ),
    region_name = "us-east-1",
    verify      = False,
)

def lire_parquet_s3(bucket: str, key: str) -> pl.DataFrame:
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return pl.read_parquet(buf)

def lister_parquet(bucket: str, prefix: str) -> list[str]:
    pages = s3.get_paginator("list_objects_v2").paginate(
        Bucket=bucket, Prefix=prefix
    )
    return sorted([
        obj["Key"]
        for page in pages
        for obj in page.get("Contents", [])
        if obj["Key"].endswith(".parquet")
    ])

def sep(titre: str = "") -> None:
    print()
    print("─" * 70)
    if titre:
        print(f"  {titre}")
        print("─" * 70)

# ============================================================
print("=" * 70)
print("DIAGNOSTIC — Anomalie effectif juin 2025")
print(f"  Mois cible  : {MOIS_CIBLE}")
print(f"  Grade cible : {GRADE_CIBLE}")
print("=" * 70)

# ============================================================
# ÉTAPE 1 : SILVER 2025 — distribution par mois
# ============================================================
sep("1. SILVER 2025 — effectifs par mois (tous grades)")

df_silver = lire_parquet_s3(BUCKET_SILVER, "panel_admin/2025.parquet")
print(f"   Fichier chargé : {len(df_silver):,} lignes × {len(df_silver.columns)} cols\n")

dist_mois = (
    df_silver
    .group_by("mois_annee")
    .agg(pl.len().alias("n_total"))
    .sort("mois_annee")
)
print(dist_mois)

# ============================================================
# ÉTAPE 2 : SILVER 2025 — effectifs A3 par mois
# ============================================================
sep(f"2. SILVER 2025 — effectifs {GRADE_CIBLE} par mois")

if "GRADE" in df_silver.columns:
    dist_grade = (
        df_silver
        .filter(pl.col("GRADE") == GRADE_CIBLE)
        .group_by("mois_annee")
        .agg(pl.len().alias(f"n_{GRADE_CIBLE}"))
        .sort("mois_annee")
    )
    print(dist_grade)
else:
    print("   ⚠️  Colonne GRADE absente du Silver 2025")

# ============================================================
# ÉTAPE 3 : SILVER 2025 — zoom sur juin 2025
# ============================================================
sep(f"3. SILVER 2025 — zoom {MOIS_CIBLE} toutes valeurs de GRADE")

df_juin_silver = df_silver.filter(pl.col("mois_annee") == MOIS_CIBLE)
n_juin_silver  = len(df_juin_silver)
print(f"   Lignes juin 2025 dans Silver : {n_juin_silver:,}\n")

if n_juin_silver > 0 and "GRADE" in df_juin_silver.columns:
    print("   Distribution par GRADE :")
    print(
        df_juin_silver["GRADE"]
        .value_counts()
        .sort("count", descending=True)
        .head(20)
    )

    print(f"\n   Colonnes SOURCE (qualité imputation) :")
    for src_col in ["montant_brut_SOURCE", "montant_net_SOURCE", "GRADE_SOURCE"]:
        if src_col in df_juin_silver.columns:
            print(f"\n   {src_col} :")
            print(
                df_juin_silver[src_col]
                .value_counts()
                .sort("count", descending=True)
            )

# ============================================================
# ÉTAPE 4 : BRONZE 2025 — juin présent ?
# ============================================================
sep("4. BRONZE 2025 — présence de juin 2025")

cles_bronze = lister_parquet(BUCKET_BRONZE, PREFIX_BRONZE)
print(f"   Fichiers Bronze trouvés : {len(cles_bronze)}")
for cle in cles_bronze:
    print(f"   · {cle}")

# Chercher les fichiers Bronze qui pourraient contenir juin 2025
# Les fichiers Bronze peuvent être nommés différemment (par mois, par organisme…)
# On cherche dans tous les fichiers 2025

cles_bronze_2025 = [c for c in cles_bronze if "2025" in c]
print(f"\n   Fichiers Bronze 2025 : {len(cles_bronze_2025)}")

for cle in cles_bronze_2025:
    print(f"\n   [{os.path.basename(cle)}]")
    try:
        df_b = lire_parquet_s3(BUCKET_BRONZE, cle)

        # Chercher une colonne mois_annee ou équivalent
        col_periode = next(
            (c for c in df_b.columns if "mois" in c.lower() or "periode" in c.lower() or "annee" in c.lower()),
            None
        )

        if col_periode:
            dist = (
                df_b[col_periode]
                .value_counts()
                .sort(col_periode)
            )
            print(f"   Colonne période : {col_periode}")
            print(dist)

            # Juin 2025 présent ?
            valeurs = df_b[col_periode].drop_nulls().unique().to_list()
            juin_present = any("06" in str(v) and "2025" in str(v) for v in valeurs)
            print(f"   → Juin 2025 détecté : {'✓ OUI' if juin_present else '✗ NON'}")
        else:
            print(f"   Colonnes disponibles : {df_b.columns[:10]}")

        del df_b
    except Exception as e:
        print(f"   ✗ Erreur lecture : {e}")

# ============================================================
# ÉTAPE 5 : GOLD panel_2025 — cohérence avec Silver
# ============================================================
sep("5. GOLD panel_2025 — effectifs A3 par mois")

try:
    df_gold = lire_parquet_s3(BUCKET_GOLD, "panel_admin/panel_2025.parquet")
    print(f"   Fichier chargé : {len(df_gold):,} lignes × {len(df_gold.columns)} cols\n")

    if "GRADE" in df_gold.columns:
        dist_gold = (
            df_gold
            .filter(pl.col("GRADE") == GRADE_CIBLE)
            .group_by("mois_annee")
            .agg(pl.len().alias(f"n_{GRADE_CIBLE}_gold"))
            .sort("mois_annee")
        )
        print(dist_gold)
    else:
        print("   ⚠️  Colonne GRADE absente du Gold panel_2025")
        print(f"   Colonnes disponibles : {df_gold.columns}")

    del df_gold
except Exception as e:
    print(f"   ✗ Erreur lecture Gold : {e}")

# ============================================================
# ÉTAPE 6 : COMPARAISON Silver vs Gold pour juin 2025
# ============================================================
sep("6. Comparaison Silver vs Gold — juin 2025 tous grades")

try:
    df_gold_juin = lire_parquet_s3(BUCKET_GOLD, "panel_admin/panel_2025.parquet")
    df_gold_juin = df_gold_juin.filter(pl.col("mois_annee") == MOIS_CIBLE)

    n_silver_juin = len(df_juin_silver)
    n_gold_juin   = len(df_gold_juin)

    print(f"   Silver juin 2025 : {n_silver_juin:,} lignes")
    print(f"   Gold   juin 2025 : {n_gold_juin:,} lignes")

    if n_silver_juin != n_gold_juin:
        print(f"   ⚠️  Écart : {abs(n_silver_juin - n_gold_juin):,} lignes manquantes dans le Gold")
    else:
        print("   ✓ Cohérent — le problème est en amont du Silver (Bronze ou fichier source)")

    del df_gold_juin
except Exception as e:
    print(f"   ✗ Erreur : {e}")

# ============================================================
# RÉSUMÉ
# ============================================================
sep()
print("=" * 70)
print("RÉSUMÉ DIAGNOSTIC")
print("=" * 70)
print(f"""
Vérifications effectuées :
  1. Distribution Silver 2025 par mois (tous grades)
  2. Distribution Silver 2025 par mois (grade {GRADE_CIBLE})
  3. Zoom juin 2025 dans Silver — distribution GRADE + colonnes SOURCE
  4. Présence de juin 2025 dans le Bronze
  5. Distribution Gold panel_2025 (grade {GRADE_CIBLE})
  6. Comparaison Silver vs Gold pour juin 2025

Interprétation :
  Si étape 3 → {MOIS_CIBLE} présent normalement dans Silver
    → Problème dans 06_compiler_panel.py ou 07_calcul_indicateur.py
  Si étape 3 → {MOIS_CIBLE} sous-représenté dans Silver
    → Problème dans 03_bronze_to_silver.py (filtre activité)
  Si étape 4 → juin 2025 absent du Bronze
    → Fichier source manquant ou mal nommé en amont
""")

DIAGNOSTIC — Anomalie effectif juin 2025
  Mois cible  : 062025
  Grade cible : A3

──────────────────────────────────────────────────────────────────────
  1. SILVER 2025 — effectifs par mois (tous grades)
──────────────────────────────────────────────────────────────────────
   Fichier chargé : 3,882,863 lignes × 241 cols

shape: (12, 2)
┌────────────┬─────────┐
│ mois_annee ┆ n_total │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ 012025     ┆ 319736  │
│ 022025     ┆ 324438  │
│ 032025     ┆ 328074  │
│ 042025     ┆ 324438  │
│ 052025     ┆ 330693  │
│ …          ┆ …       │
│ 082025     ┆ 337240  │
│ 092025     ┆ 338488  │
│ 102025     ┆ 339702  │
│ 112025     ┆ 339807  │
│ 122025     ┆ 338713  │
└────────────┴─────────┘

──────────────────────────────────────────────────────────────────────
  2. SILVER 2025 — effectifs A3 par mois
──────────────────────────────────────────────────────────────────────
shape: (12, 2)
┌────────────┬───────┐
│ mois_anne

In [2]:
# ============================================================
# PANEL ADMIN — TABLEAU GRADE PAR MOIS (2024-2025)
# ============================================================
# Lit les fichiers Gold panel_2024 et panel_2025,
# calcule les statistiques de revenu par GRADE × mois,
# et produit un fichier Excel formaté avec :
#   - Un onglet "Tous grades" (vue consolidée)
#   - Un onglet par grade
#   - Mise en forme professionnelle
#
# Sortie : grade_par_mois_2024_2025.xlsx (local + staging)
#
# Dépendances :
#   pip install polars openpyxl boto3 python-dotenv
# ============================================================

import io
import os
from dotenv import load_dotenv
import boto3
from botocore.client import Config
import polars as pl
import openpyxl
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side, GradientFill
)
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

load_dotenv(".env")

# ============================================================
# CONFIGURATION
# ============================================================

MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT",   "http://192.168.1.230:30137")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "datalab-team")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minio-datalabteam123")

BUCKET_GOLD    = os.getenv("BUCKET_GOLD",    "gold")
BUCKET_STAGING = os.getenv("BUCKET_STAGING", "staging")

ANNEES       = [2024, 2025]
OUTPUT_LOCAL = "grade_par_mois_2024_2025.xlsx"
OUTPUT_S3    = "panel_admin/exports_gold/grade_par_mois_2024_2025.xlsx"

MOIS_LABELS = {
    1: "Janvier", 2: "Février",   3: "Mars",
    4: "Avril",   5: "Mai",       6: "Juin",
    7: "Juillet", 8: "Août",      9: "Septembre",
    10: "Octobre", 11: "Novembre", 12: "Décembre",
}

# ============================================================
# CLIENT S3
# ============================================================

s3 = boto3.client(
    "s3",
    endpoint_url          = MINIO_ENDPOINT,
    aws_access_key_id     = MINIO_ACCESS_KEY,
    aws_secret_access_key = MINIO_SECRET_KEY,
    config                = Config(
        signature_version = "s3v4",
        retries           = {"max_attempts": 5, "mode": "adaptive"},
        connect_timeout   = 30,
        read_timeout      = 120,
    ),
    region_name = "us-east-1",
    verify      = False,
)

def lire_parquet_s3(bucket: str, key: str) -> pl.DataFrame:
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return pl.read_parquet(buf)

# ============================================================
# CHARGEMENT ET CALCUL DES STATISTIQUES
# ============================================================

print("=" * 70)
print("TABLEAU GRADE PAR MOIS — 2024-2025")
print("=" * 70)
print()

frames = []
for annee in ANNEES:
    cle = f"panel_admin/panel_{annee}.parquet"
    print(f"  Lecture {cle}...", end=" ", flush=True)
    df = lire_parquet_s3(BUCKET_GOLD, cle)
    frames.append(df)
    print(f"✓  {len(df):,} lignes")

panel = pl.concat(frames, how="diagonal")
print(f"\n  Total : {len(panel):,} lignes\n")

# Calcul des stats par GRADE × année × mois
stats = (
    panel
    .filter(
        pl.col("annee").is_in(ANNEES) &
        pl.col("GRADE").is_not_null() &
        pl.col("montant_brut").is_not_null()
    )
    .group_by(["annee", "mois", "GRADE"])
    .agg([
        pl.len().alias("Effectif"),
        pl.col("montant_brut").mean().round(0).alias("Revenu_moyen"),
        pl.col("montant_brut").median().round(0).alias("Revenu_median"),
        pl.col("montant_brut").min().round(0).alias("Revenu_min"),
        pl.col("montant_brut").max().round(0).alias("Revenu_max"),
        pl.col("montant_brut").quantile(0.25, interpolation="nearest").round(0).alias("Revenu_p25"),
        pl.col("montant_brut").quantile(0.75, interpolation="nearest").round(0).alias("Revenu_p75"),
    ])
    .with_columns(
        pl.col("mois").map_elements(lambda m: MOIS_LABELS.get(m, str(m)), return_dtype=pl.Utf8).alias("Mois_label")
    )
    .sort(["annee", "mois", "GRADE"])
)

grades       = sorted(stats["GRADE"].drop_nulls().unique().to_list())
n_grades     = len(grades)
n_lignes     = len(stats)
print(f"  Grades distincts : {n_grades}")
print(f"  Lignes de stats  : {n_lignes:,}\n")

# Conversion en pandas pour openpyxl
import pandas as pd
df_stats = stats.to_pandas()

# ============================================================
# STYLES OPENPYXL
# ============================================================

BLEU_FONCE  = "1F3864"
BLEU_MOYEN  = "2E75B6"
BLEU_CLAIR  = "D6E4F0"
BLEU_TRES_CLAIR = "EBF5FB"
GRIS_CLAIR  = "F2F2F2"
BLANC       = "FFFFFF"
VERT        = "70AD47"
ORANGE      = "ED7D31"

def style_entete_principal(cell, bg=BLEU_FONCE):
    cell.font      = Font(name="Arial", bold=True, color=BLANC, size=11)
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

def style_entete_col(cell):
    cell.font      = Font(name="Arial", bold=True, color=BLANC, size=10)
    cell.fill      = PatternFill("solid", fgColor=BLEU_MOYEN)
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border    = bordure_fine()

def style_annee(cell):
    cell.font      = Font(name="Arial", bold=True, color=BLANC, size=10)
    cell.fill      = PatternFill("solid", fgColor=BLEU_MOYEN)
    cell.alignment = Alignment(horizontal="center", vertical="center")

def style_mois(cell, pair=True):
    cell.font      = Font(name="Arial", bold=True, color=BLEU_FONCE, size=10)
    cell.fill      = PatternFill("solid", fgColor=BLEU_CLAIR if pair else BLEU_TRES_CLAIR)
    cell.alignment = Alignment(horizontal="left", vertical="center", indent=1)

def style_data(cell, pair=True):
    cell.font      = Font(name="Arial", size=10)
    cell.fill      = PatternFill("solid", fgColor=BLANC if pair else GRIS_CLAIR)
    cell.alignment = Alignment(horizontal="right", vertical="center")
    cell.border    = bordure_fine()

def style_effectif(cell, pair=True):
    cell.font      = Font(name="Arial", bold=True, size=10, color=BLEU_FONCE)
    cell.fill      = PatternFill("solid", fgColor=BLANC if pair else GRIS_CLAIR)
    cell.alignment = Alignment(horizontal="right", vertical="center")
    cell.border    = bordure_fine()

def bordure_fine():
    c = Side(style="thin", color="BFBFBF")
    return Border(left=c, right=c, top=c, bottom=c)

def bordure_epaisse():
    c = Side(style="medium", color=BLEU_MOYEN)
    return Border(left=c, right=c, top=c, bottom=c)

FMT_ENTIER   = "#,##0"
FMT_MONNAIE  = "#,##0"

# ============================================================
# CRÉATION DU WORKBOOK
# ============================================================

wb = openpyxl.Workbook()
wb.remove(wb.active)  # supprimer la feuille vide par défaut

COLS_DISPLAY = [
    ("Année",         "annee",         10,  False),
    ("Mois",          "Mois_label",    13,  False),
    ("Grade",         "GRADE",         10,  False),
    ("Effectif",      "Effectif",      11,  True),
    ("Moy. Brut",     "Revenu_moyen",  14,  True),
    ("Médiane",       "Revenu_median", 14,  True),
    ("Min",           "Revenu_min",    13,  True),
    ("Max",           "Revenu_max",    13,  True),
    ("P25",           "Revenu_p25",    13,  True),
    ("P75",           "Revenu_p75",    13,  True),
]

def ecrire_onglet(ws, df_onglet, titre_onglet, show_grade_col=True):
    """Écrit un onglet formaté. show_grade_col=False pour les onglets par grade."""

    cols = [c for c in COLS_DISPLAY if (c[2] != 10 or c[1] != "GRADE" or show_grade_col)]
    n_cols = len(cols)

    # ── Titre ──────────────────────────────────────────────
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols)
    titre_cell = ws.cell(row=1, column=1, value=titre_onglet)
    titre_cell.font      = Font(name="Arial", bold=True, color=BLANC, size=13)
    titre_cell.fill      = PatternFill("solid", fgColor=BLEU_FONCE)
    titre_cell.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 28

    # ── En-têtes colonnes ──────────────────────────────────
    for j, (label, _, largeur, _) in enumerate(cols, 1):
        cell = ws.cell(row=2, column=j, value=label)
        style_entete_col(cell)
        ws.column_dimensions[get_column_letter(j)].width = largeur
    ws.row_dimensions[2].height = 20

    # ── Données ────────────────────────────────────────────
    annee_courante = None
    mois_courant   = None
    ligne_excel    = 3
    pair           = True

    for _, row in df_onglet.iterrows():
        annee = int(row["annee"])
        mois  = row["Mois_label"]

        # Alternance de couleur à chaque changement de mois
        if mois != mois_courant:
            pair = not pair
            mois_courant = mois

        for j, (_, col_key, _, is_numeric) in enumerate(cols, 1):
            val  = row.get(col_key, "")
            cell = ws.cell(row=ligne_excel, column=j)

            if col_key == "annee":
                cell.value = int(val) if pd.notna(val) else ""
                style_annee(cell) if annee != annee_courante else style_data(cell, pair)
                if annee != annee_courante:
                    annee_courante = annee
            elif col_key == "Mois_label":
                cell.value = val
                style_mois(cell, pair)
            elif col_key == "Effectif":
                cell.value = int(val) if pd.notna(val) else None
                cell.number_format = FMT_ENTIER
                style_effectif(cell, pair)
            elif is_numeric:
                cell.value = int(val) if pd.notna(val) else None
                cell.number_format = FMT_MONNAIE
                style_data(cell, pair)
            else:
                cell.value = val
                style_data(cell, pair)

        ws.row_dimensions[ligne_excel].height = 16
        ligne_excel += 1

    # ── Figer les volets ───────────────────────────────────
    ws.freeze_panes = "A3"

    # ── Filtre automatique ─────────────────────────────────
    ws.auto_filter.ref = f"A2:{get_column_letter(n_cols)}{ligne_excel - 1}"

    # ── Bordure extérieure du tableau ──────────────────────
    from openpyxl.styles import Border, Side
    ep = Side(style="medium", color=BLEU_MOYEN)
    for row in ws.iter_rows(min_row=1, max_row=ligne_excel - 1, min_col=1, max_col=n_cols):
        for cell in row:
            b = cell.border
            new_b = Border(
                left   = ep if cell.column == 1             else b.left,
                right  = ep if cell.column == n_cols        else b.right,
                top    = ep if cell.row == 1                else b.top,
                bottom = ep if cell.row == ligne_excel - 1  else b.bottom,
            )
            cell.border = new_b

    # ── Mise en forme conditionnelle sur Effectif ──────────
    col_eff = next(j for j, (_, k, _, _) in enumerate(cols, 1) if k == "Effectif")
    col_eff_lettre = get_column_letter(col_eff)
    plage_eff = f"{col_eff_lettre}3:{col_eff_lettre}{ligne_excel - 1}"
    ws.conditional_formatting.add(
        plage_eff,
        ColorScaleRule(
            start_type="min", start_color="FFFFFF",
            end_type="max",   end_color="2E75B6",
        )
    )

    print(f"  ✓ Onglet '{ws.title}' — {ligne_excel - 3} lignes")


# ── Onglet 1 : Tous grades consolidés ─────────────────────
ws_all = wb.create_sheet("Tous grades")
ecrire_onglet(
    ws_all,
    df_stats,
    "Statistiques de revenu brut par GRADE et par mois — 2024-2025",
    show_grade_col=True,
)

# ── Onglets par grade ──────────────────────────────────────
for grade in grades:
    df_grade = df_stats[df_stats["GRADE"] == grade].copy()
    nom_onglet = f"Grade {grade}"[:31]  # Excel : max 31 caractères
    ws_g = wb.create_sheet(nom_onglet)
    ecrire_onglet(
        ws_g,
        df_grade,
        f"Grade {grade} — Statistiques de revenu brut par mois (2024-2025)",
        show_grade_col=False,
    )

# ── Onglet Sommaire ────────────────────────────────────────
ws_som = wb.create_sheet("Sommaire", 0)  # en premier

ws_som.column_dimensions["A"].width = 30
ws_som.column_dimensions["B"].width = 18
ws_som.column_dimensions["C"].width = 18

ws_som.merge_cells("A1:C1")
c = ws_som.cell(row=1, column=1, value="PANEL ADMINISTRATIF — Salaires 2024-2025")
c.font = Font(name="Arial", bold=True, color=BLANC, size=14)
c.fill = PatternFill("solid", fgColor=BLEU_FONCE)
c.alignment = Alignment(horizontal="center", vertical="center")
ws_som.row_dimensions[1].height = 32

for j, label in enumerate(["Onglet", "Grades", "Périodes"], 1):
    cell = ws_som.cell(row=2, column=j, value=label)
    style_entete_col(cell)

lignes_som = [
    ("Tous grades", f"{n_grades} grades", "Jan 2024 → Déc 2025"),
] + [
    (f"Grade {g}", g, "Jan 2024 → Déc 2025")
    for g in grades
]

for i, (onglet, grade_info, periode) in enumerate(lignes_som, 3):
    pair = i % 2 == 0
    bg   = BLANC if pair else GRIS_CLAIR
    for j, val in enumerate([onglet, grade_info, periode], 1):
        cell = ws_som.cell(row=i, column=j, value=val)
        cell.font      = Font(name="Arial", size=10)
        cell.fill      = PatternFill("solid", fgColor=bg)
        cell.alignment = Alignment(horizontal="left" if j == 1 else "center", vertical="center")
        cell.border    = bordure_fine()
    ws_som.row_dimensions[i].height = 16

ws_som.freeze_panes = "A3"
print(f"  ✓ Onglet 'Sommaire' — {len(lignes_som)} entrées")

# ============================================================
# SAUVEGARDE
# ============================================================

wb.save(OUTPUT_LOCAL)
print(f"\n  ✓ Fichier sauvegardé : {OUTPUT_LOCAL}")

# Upload vers staging
try:
    s3.upload_file(OUTPUT_LOCAL, BUCKET_STAGING, OUTPUT_S3)
    print(f"  ✓ Upload : s3://{BUCKET_STAGING}/{OUTPUT_S3}")
except Exception as e:
    print(f"  ⚠️  Upload staging ignoré : {e}")

print()
print("=" * 70)
print("✓ TERMINÉ")
print("=" * 70)
print(f"""
Fichier produit : {OUTPUT_LOCAL}
  · Onglet Sommaire     : vue d'ensemble
  · Onglet Tous grades  : {n_lignes:,} lignes — toutes combinaisons GRADE × mois
  · {n_grades} onglets par grade : {', '.join(grades[:5])}{'...' if n_grades > 5 else ''}

Colonnes : Effectif | Moy. Brut | Médiane | Min | Max | P25 | P75
""")

TABLEAU GRADE PAR MOIS — 2024-2025

  Lecture panel_admin/panel_2024.parquet... ✓  3,719,583 lignes
  Lecture panel_admin/panel_2025.parquet... ✓  3,871,319 lignes

  Total : 7,590,902 lignes

  Grades distincts : 14
  Lignes de stats  : 309

  ✓ Onglet 'Tous grades' — 309 lignes
  ✓ Onglet 'Grade A3' — 24 lignes
  ✓ Onglet 'Grade A4' — 23 lignes
  ✓ Onglet 'Grade A5' — 23 lignes
  ✓ Onglet 'Grade A6' — 23 lignes
  ✓ Onglet 'Grade A7' — 24 lignes
  ✓ Onglet 'Grade B1' — 24 lignes
  ✓ Onglet 'Grade B3' — 24 lignes
  ✓ Onglet 'Grade C1' — 24 lignes
  ✓ Onglet 'Grade C2' — 23 lignes
  ✓ Onglet 'Grade C3' — 24 lignes
  ✓ Onglet 'Grade D' — 1 lignes
  ✓ Onglet 'Grade D1' — 24 lignes
  ✓ Onglet 'Grade D2' — 24 lignes
  ✓ Onglet 'Grade NF' — 24 lignes
  ✓ Onglet 'Sommaire' — 15 entrées

  ✓ Fichier sauvegardé : grade_par_mois_2024_2025.xlsx
  ✓ Upload : s3://staging/panel_admin/exports_gold/grade_par_mois_2024_2025.xlsx

✓ TERMINÉ

Fichier produit : grade_par_mois_2024_2025.xlsx
  · Onglet So

In [3]:
# ============================================================
# EXPORT EXCEL — Effectifs par GRADE × mois (2024 & 2025)
# ============================================================

import io
import os
from dotenv import load_dotenv
import boto3
from botocore.client import Config
import polars as pl
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

load_dotenv(".env")

MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT",   "http://192.168.1.230:30137")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "datalab-team")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minio-datalabteam123")
BUCKET_GOLD      = os.getenv("BUCKET_GOLD",      "gold")
BUCKET_STAGING   = os.getenv("BUCKET_STAGING",   "staging")

ANNEES = [2024, 2025]

MOIS_LABELS = {
    1:"Janvier", 2:"Février", 3:"Mars", 4:"Avril",
    5:"Mai", 6:"Juin", 7:"Juillet", 8:"Août",
    9:"Septembre", 10:"Octobre", 11:"Novembre", 12:"Décembre",
}

s3 = boto3.client(
    "s3",
    endpoint_url          = MINIO_ENDPOINT,
    aws_access_key_id     = MINIO_ACCESS_KEY,
    aws_secret_access_key = MINIO_SECRET_KEY,
    config                = Config(signature_version="s3v4", region_name="us-east-1"),
    verify                = False,
)

def lire_parquet_s3(bucket, key):
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return pl.read_parquet(buf)

# ── Chargement ─────────────────────────────────────────────
print("Chargement Gold...")
frames = []
for annee in ANNEES:
    df = lire_parquet_s3(BUCKET_GOLD, f"panel_admin/panel_{annee}.parquet")
    frames.append(df)
    print(f"  {annee} : {len(df):,} lignes")
panel = pl.concat(frames, how="diagonal")

# ── Calcul des effectifs ────────────────────────────────────
effectifs = (
    panel
    .filter(pl.col("GRADE").is_not_null())
    .group_by(["annee", "mois", "GRADE"])
    .agg(pl.len().alias("effectif"))
    .sort(["annee", "mois", "GRADE"])
)

grades = sorted(effectifs["GRADE"].unique().to_list())
print(f"\nGrades : {grades}")

# ── Styles ──────────────────────────────────────────────────
BLEU_FONCE  = "1F3864"
BLEU_MOYEN  = "2E75B6"
BLEU_CLAIR  = "D6E4F0"
GRIS        = "F2F2F2"
BLANC       = "FFFFFF"
ORANGE      = "FFC000"

def ep(): s = Side(style="medium", color=BLEU_MOYEN);  return Border(left=s, right=s, top=s, bottom=s)
def fin(): s = Side(style="thin",  color="BFBFBF");    return Border(left=s, right=s, top=s, bottom=s)

def h_titre(cell, texte, bg=BLEU_FONCE):
    cell.value     = texte
    cell.font      = Font(name="Arial", bold=True, color=BLANC, size=12)
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal="center", vertical="center")

def h_col(cell, texte):
    cell.value     = texte
    cell.font      = Font(name="Arial", bold=True, color=BLANC, size=10)
    cell.fill      = PatternFill("solid", fgColor=BLEU_MOYEN)
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border    = fin()

def h_mois(cell, texte, pair):
    cell.value     = texte
    cell.font      = Font(name="Arial", bold=True, color=BLEU_FONCE, size=10)
    cell.fill      = PatternFill("solid", fgColor=BLEU_CLAIR if pair else BLANC)
    cell.alignment = Alignment(horizontal="left", vertical="center", indent=1)
    cell.border    = fin()

def d_cell(cell, valeur, pair, anomalie=False):
    cell.value          = valeur
    cell.number_format  = "#,##0"
    cell.font           = Font(name="Arial", size=10,
                               color="CC0000" if anomalie else "000000")
    cell.fill           = PatternFill("solid", fgColor=GRIS if not pair else BLANC)
    cell.alignment      = Alignment(horizontal="right", vertical="center")
    cell.border         = fin()

# ── Workbook ────────────────────────────────────────────────
wb   = openpyxl.Workbook()
wb.remove(wb.active)

# ── Un onglet par année + un onglet comparatif ──────────────
def ecrire_onglet_annee(annee):
    ws  = wb.create_sheet(str(annee))
    n_g = len(grades)
    n_c = 1 + n_g + 1          # mois | grade1…gradeN | Total

    # Titre
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_c)
    h_titre(ws.cell(1, 1), f"Effectifs par GRADE — {annee}", BLEU_FONCE)
    ws.row_dimensions[1].height = 28

    # En-têtes colonnes
    h_col(ws.cell(2, 1), "Mois")
    for j, g in enumerate(grades, 2):
        h_col(ws.cell(2, j), g)
    h_col(ws.cell(2, n_c), "TOTAL")
    ws.row_dimensions[2].height = 18

    # Filtre les données de l'année
    df_an = effectifs.filter(pl.col("annee") == annee)

    # Pivot en dict (mois, grade) → effectif
    pivot = {}
    for row in df_an.iter_rows(named=True):
        pivot[(row["mois"], row["GRADE"])] = row["effectif"]

    # Écriture ligne par ligne
    for i, mois_num in enumerate(range(1, 13), 3):
        pair = i % 2 == 0
        h_mois(ws.cell(i, 1), MOIS_LABELS[mois_num], pair)

        total_mois = 0
        for j, g in enumerate(grades, 2):
            val = pivot.get((mois_num, g), None)
            # Anomalie = effectif < 10% de la médiane du grade sur l'année
            vals_grade = [v for (m, gr), v in pivot.items() if gr == g]
            mediane_grade = sorted(vals_grade)[len(vals_grade)//2] if vals_grade else 1
            anomalie = val is not None and val < mediane_grade * 0.2
            d_cell(ws.cell(i, j), val, pair, anomalie)
            total_mois += val or 0

        # Cellule total
        tc = ws.cell(i, n_c)
        tc.value          = total_mois
        tc.number_format  = "#,##0"
        tc.font           = Font(name="Arial", bold=True, size=10)
        tc.fill           = PatternFill("solid", fgColor=BLEU_CLAIR)
        tc.alignment      = Alignment(horizontal="right", vertical="center")
        tc.border         = fin()
        ws.row_dimensions[i].height = 16

    # Ligne total annuel
    row_tot = 15
    tc = ws.cell(row_tot, 1)
    tc.value     = "TOTAL ANNUEL"
    tc.font      = Font(name="Arial", bold=True, color=BLANC, size=10)
    tc.fill      = PatternFill("solid", fgColor=BLEU_MOYEN)
    tc.alignment = Alignment(horizontal="left", vertical="center", indent=1)
    tc.border    = fin()

    for j, g in enumerate(grades, 2):
        col_lettre = get_column_letter(j)
        tc = ws.cell(row_tot, j)
        tc.value         = f"=SUM({col_lettre}3:{col_lettre}14)"
        tc.number_format = "#,##0"
        tc.font          = Font(name="Arial", bold=True, color=BLANC, size=10)
        tc.fill          = PatternFill("solid", fgColor=BLEU_MOYEN)
        tc.alignment     = Alignment(horizontal="right", vertical="center")
        tc.border        = fin()

    col_tot = get_column_letter(n_c)
    tc = ws.cell(row_tot, n_c)
    tc.value         = f"=SUM({col_tot}3:{col_tot}14)"
    tc.number_format = "#,##0"
    tc.font          = Font(name="Arial", bold=True, color=BLANC, size=10)
    tc.fill          = PatternFill("solid", fgColor=BLEU_MOYEN)
    tc.alignment     = Alignment(horizontal="right", vertical="center")
    tc.border        = fin()
    ws.row_dimensions[row_tot].height = 18

    # Largeurs colonnes
    ws.column_dimensions["A"].width = 14
    for j in range(2, n_c + 1):
        ws.column_dimensions[get_column_letter(j)].width = 13

    # Mise en forme conditionnelle (dégradé blanc→bleu) sur les données
    for j in range(2, n_c):
        col_l = get_column_letter(j)
        ws.conditional_formatting.add(
            f"{col_l}3:{col_l}14",
            ColorScaleRule(
                start_type="min", start_color="FFFFFF",
                end_type="max",   end_color="2E75B6",
            )
        )

    # Volets figés
    ws.freeze_panes = "B3"
    print(f"  ✓ Onglet {annee}")

# Écrire les deux onglets annuels
for annee in ANNEES:
    ecrire_onglet_annee(annee)

# ── Onglet comparatif 2024 vs 2025 ─────────────────────────
ws_c = wb.create_sheet("Comparaison 2024-2025")
n_g  = len(grades)
n_c  = 1 + n_g * 2 + 1   # mois | G1_2024 G1_2025 … | total

# Titre
ws_c.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_c)
h_titre(ws_c.cell(1, 1), "Comparaison effectifs par GRADE — 2024 vs 2025", BLEU_FONCE)
ws_c.row_dimensions[1].height = 28

# Ligne sous-titre années
ws_c.merge_cells(start_row=2, start_column=2, end_row=2, end_column=1 + n_g)
c = ws_c.cell(2, 2, "2024")
c.font = Font(name="Arial", bold=True, color=BLANC, size=10)
c.fill = PatternFill("solid", fgColor="2E75B6")
c.alignment = Alignment(horizontal="center")

ws_c.merge_cells(start_row=2, start_column=2 + n_g, end_row=2, end_column=1 + n_g * 2)
c = ws_c.cell(2, 2 + n_g, "2025")
c.font = Font(name="Arial", bold=True, color=BLANC, size=10)
c.fill = PatternFill("solid", fgColor="70AD47")
c.alignment = Alignment(horizontal="center")

# En-têtes grades
h_col(ws_c.cell(3, 1), "Mois")
for j, g in enumerate(grades, 2):
    h_col(ws_c.cell(3, j),        g)          # 2024
    h_col(ws_c.cell(3, j + n_g),  g)          # 2025
h_col(ws_c.cell(3, n_c), "Δ Total")
ws_c.row_dimensions[3].height = 18

# Pivots
pivot_2024 = {(r["mois"], r["GRADE"]): r["effectif"]
              for r in effectifs.filter(pl.col("annee") == 2024).iter_rows(named=True)}
pivot_2025 = {(r["mois"], r["GRADE"]): r["effectif"]
              for r in effectifs.filter(pl.col("annee") == 2025).iter_rows(named=True)}

for i, mois_num in enumerate(range(1, 13), 4):
    pair = i % 2 == 0
    h_mois(ws_c.cell(i, 1), MOIS_LABELS[mois_num], pair)

    tot_24 = tot_25 = 0
    for j, g in enumerate(grades, 2):
        v24 = pivot_2024.get((mois_num, g), None)
        v25 = pivot_2025.get((mois_num, g), None)
        d_cell(ws_c.cell(i, j),        v24, pair)
        d_cell(ws_c.cell(i, j + n_g),  v25, pair)
        tot_24 += v24 or 0
        tot_25 += v25 or 0

    # Δ Total
    delta = tot_25 - tot_24
    dc = ws_c.cell(i, n_c)
    dc.value         = delta
    dc.number_format = "+#,##0;-#,##0;0"
    dc.font          = Font(name="Arial", bold=True, size=10,
                            color="375623" if delta >= 0 else "CC0000")
    dc.fill          = PatternFill("solid", fgColor=BLANC if pair else GRIS)
    dc.alignment     = Alignment(horizontal="right", vertical="center")
    dc.border        = fin()
    ws_c.row_dimensions[i].height = 16

ws_c.column_dimensions["A"].width = 14
for j in range(2, n_c + 1):
    ws_c.column_dimensions[get_column_letter(j)].width = 12
ws_c.freeze_panes = "B4"
print("  ✓ Onglet Comparaison 2024-2025")

# ── Sauvegarde ──────────────────────────────────────────────
OUTPUT = "effectifs_grade_par_mois_2024_2025.xlsx"
wb.save(OUTPUT)
print(f"\n✓ Fichier : {OUTPUT}")

try:
    s3.upload_file(OUTPUT, BUCKET_STAGING,
                   "panel_admin/exports_gold/effectifs_grade_par_mois_2024_2025.xlsx")
    print(f"✓ Upload staging OK")
except Exception as e:
    print(f"⚠️  Upload staging : {e}")

Chargement Gold...
  2024 : 3,719,583 lignes
  2025 : 3,871,319 lignes

Grades : ['A3', 'A4', 'A5', 'A6', 'A7', 'B1', 'B3', 'C1', 'C2', 'C3', 'D', 'D1', 'D2', 'NF']
  ✓ Onglet 2024
  ✓ Onglet 2025
  ✓ Onglet Comparaison 2024-2025

✓ Fichier : effectifs_grade_par_mois_2024_2025.xlsx
✓ Upload staging OK
